In [10]:
import sys
import math
import time
from pathlib import Path

import torch
import torch.nn as nn
import pandas as pd
from tqdm.auto import tqdm

root = Path.cwd()
if root.name in ["train", "notebooks"]:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data.dataset import create_dataloaders
from src.models.cnn import CNNLanguageModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


Cấu hình

In [11]:
# CẤU HÌNH THÍ NGHIỆM
CONTEXT_LENGTH = 64
BATCH_SIZE = 32
EPOCHS = 10                 # Tổng số epoch (kể cả đã train trước đó)
PATIENCE = 5
LEARNING_RATE = 1e-3
MAX_GRAD_NORM = 1.0

MODEL_NAME = "cnn"
EXPERIMENT_NAME = f"word_c{CONTEXT_LENGTH}"

# Tiếp tục từ checkpoint cuối cùng (nếu muốn train lại từ đầu: False)
RESUME_TRAINING = True
RESET_PATIENCE_ON_RESUME = True

# Ghi log TensorBoard (cần cài tensorboard nếu bật True)
USE_TENSORBOARD = False

Load Dataset

In [12]:
train_loader, valid_loader, test_loader, vocab_size = (
    create_dataloaders(
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE
    )
)

print("Vocabulary size:", vocab_size)
print("Train batches:", len(train_loader))

Vocabulary size: 10000
Train batches: 1042


Khởi tạo CNN

In [13]:
model = CNNLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=256,
    hidden_dim=256,
    kernel_size=3,
    dropout=0.2,
    dilations=(1, 2, 4)
).to(device)

total_params = sum(p.numel() for p in model.parameters())
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Parameters:", f"{total_params:,}")
print("Receptive field:", model.receptive_field)

Parameters: 5,720,592
Receptive field: 15


## Huấn luyện CNN (PyTorch training loop)

Quá trình huấn luyện nằm **trực tiếp trong notebook**: `for epoch` → Train từng batch → Validation → Ghi log → Lưu `last_checkpoint.pt` và `best_model.pt`. Không gọi `fit()` hay `evaluate()`.

- `last_checkpoint.pt`: đầy đủ model, optimizer, epoch, history để train tiếp.
- `best_model.pt`: `state_dict` của model đạt validation loss thấp nhất (giữ tương thích với phần Test).
- Khi resume, đặt `RESUME_TRAINING=True` và tăng `EPOCHS` thành **tổng** epoch mong muốn.

In [14]:
# =============================================================
# 1. CHECKPOINTS + LOGS
# =============================================================
checkpoint_dir = root / "checkpoints" / MODEL_NAME / EXPERIMENT_NAME
results_dir = root / "results" / MODEL_NAME / EXPERIMENT_NAME
checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

best_path = checkpoint_dir / "best_model.pt"
last_path = checkpoint_dir / "last_checkpoint.pt"

writer = None
if USE_TENSORBOARD:
    from torch.utils.tensorboard import SummaryWriter
    writer = SummaryWriter(log_dir=str(results_dir / "tensorboard"))

# =============================================================
# 2. RESUME TRAINING (NẾU CÓ)
# =============================================================
start_epoch = 0
best_valid_loss = float("inf")
patience_counter = 0
history = []
previous_training_time = 0.0

if RESUME_TRAINING:
    if not last_path.exists():
        raise FileNotFoundError(f"Checkpoint không tồn tại: {last_path}")

    # Chỉ load các checkpoint do bạn tự tạo / tin tưởng.
    checkpoint = torch.load(last_path, map_location=device, weights_only=False)
    if (checkpoint["context_length"] != CONTEXT_LENGTH
            or checkpoint["batch_size"] != BATCH_SIZE
            or checkpoint["vocab_size"] != vocab_size):
        raise ValueError("CONTEXT_LENGTH, BATCH_SIZE hoặc vocab_size khác checkpoint cũ")

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"]
    best_valid_loss = checkpoint["best_valid_loss"]
    patience_counter = checkpoint["patience_counter"]
    history = checkpoint["history"]
    previous_training_time = checkpoint.get("training_time_seconds", 0.0)

    if RESET_PATIENCE_ON_RESUME:
        patience_counter = 0

    print(f"Resume từ epoch {start_epoch + 1} đến epoch {EPOCHS}")
else:
    print("Train từ đầu. Hãy chạy lại ô khởi tạo model/optimizer khi muốn train mới.")

if start_epoch >= EPOCHS:
    print(f"Checkpoint đã hoàn thành {start_epoch} epoch; EPOCHS={EPOCHS}, không cần train thêm.")

# =============================================================
# 3. TRAIN + VALIDATION + SAVE (TRONG CÙNG VÒNG EPOCH)
# =============================================================
if device.type == "cuda":
    torch.cuda.synchronize()
start_time = time.time()

for epoch in range(start_epoch, EPOCHS):

    # ------------------ TRAINING PHASE ------------------
    model.train()
    train_total_loss = 0.0
    train_total_tokens = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} - Train", colour="cyan")

    for batch_idx, (X, Y) in enumerate(train_bar):
        X = X.to(device)
        Y = Y.to(device)

        # Forward
        logits = model(X)
        loss = criterion(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))

        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()

        n_tokens = Y.numel()
        train_total_loss += loss.item() * n_tokens
        train_total_tokens += n_tokens
        mean_train_loss = train_total_loss / train_total_tokens
        train_bar.set_postfix(loss=f"{mean_train_loss:.4f}")

        if writer is not None:
            writer.add_scalar(
                "Train/Batch_Loss", loss.item(),
                epoch * len(train_loader) + batch_idx
            )

    train_loss = train_total_loss / train_total_tokens
    train_ppl = math.exp(train_loss)

    # ------------------ VALIDATION PHASE ------------------
    model.eval()
    valid_total_loss = 0.0
    valid_total_tokens = 0

    with torch.no_grad():
        val_bar = tqdm(valid_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} - Val", colour="yellow")
        for X, Y in val_bar:
            X = X.to(device)
            Y = Y.to(device)

            logits = model(X)
            loss = criterion(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))

            n_tokens = Y.numel()
            valid_total_loss += loss.item() * n_tokens
            valid_total_tokens += n_tokens
            val_bar.set_postfix(loss=f"{valid_total_loss / valid_total_tokens:.4f}")

    valid_loss = valid_total_loss / valid_total_tokens
    valid_ppl = math.exp(valid_loss)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Train PPL: {train_ppl:.2f} | "
        f"Valid Loss: {valid_loss:.4f} | Valid PPL: {valid_ppl:.2f}"
    )

    # ------------------ LOGGING ------------------
    history.append({
        "epoch": epoch + 1,
        "context_length": CONTEXT_LENGTH,
        "batch_size": BATCH_SIZE,
        "train_loss": train_loss,
        "train_ppl": train_ppl,
        "valid_loss": valid_loss,
        "valid_ppl": valid_ppl,
    })
    pd.DataFrame(history).to_csv(results_dir / "history.csv", index=False)

    if writer is not None:
        writer.add_scalar("Train/Loss", train_loss, epoch + 1)
        writer.add_scalar("Train/PPL", train_ppl, epoch + 1)
        writer.add_scalar("Val/Loss", valid_loss, epoch + 1)
        writer.add_scalar("Val/PPL", valid_ppl, epoch + 1)

    # ------------------ SAVE BEST MODEL ------------------
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        patience_counter = 0
        # best chỉ chứa weights để ô Test / Inference load như cũ
        torch.save(model.state_dict(), best_path)
        print(f"  Saved BEST: {best_path}")
    else:
        patience_counter += 1
        print(f"  No improvement: {patience_counter}/{PATIENCE}")

    # ------------------ SAVE LAST CHECKPOINT ------------------
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_time = previous_training_time + time.time() - start_time

    checkpoint = {
        "epoch": epoch + 1,   # số epoch đã hoàn thành
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_valid_loss": best_valid_loss,
        "patience_counter": patience_counter,
        "history": history,
        "training_time_seconds": elapsed_time,
        "context_length": CONTEXT_LENGTH,
        "batch_size": BATCH_SIZE,
        "vocab_size": vocab_size,
    }
    torch.save(checkpoint, last_path)
    print(f"  Saved LAST: {last_path}")

    # ------------------ EARLY STOPPING ------------------
    if patience_counter >= PATIENCE:
        print("Early Stopping!")
        break

if device.type == "cuda":
    torch.cuda.synchronize()
training_time = previous_training_time + time.time() - start_time
if writer is not None:
    writer.close()
print(f"Tổng training time: {training_time / 60:.2f} phút")

# Giữ tương thích với các ô Test và Summary ở phía dưới
result = {
    "best_valid_loss": best_valid_loss,
    "best_valid_ppl": math.exp(best_valid_loss),
    "epochs_trained": len(history),
    "training_time_seconds": training_time,
    "checkpoint_path": str(best_path),
    "last_checkpoint_path": str(last_path),
    "results_dir": str(results_dir),
    "history": pd.DataFrame(history),
}

Resume từ epoch 3 đến epoch 10


Epoch 3/10 - Train:   0%|          | 0/1042 [00:00<?, ?it/s]

KeyboardInterrupt: 

Test mô hình tốt nhất

In [17]:
# Nạp checkpoint tốt nhất, đánh giá trên tập test (không cập nhật trọng số)
model.load_state_dict(torch.load(r"C:\nam4\Nam4\deeplearning\baitapnhom\checkpoints\cnn\word_c64\best_model.pt", map_location=device, weights_only=True))
model.eval()
test_total_loss = 0.0
test_total_tokens = 0

with torch.no_grad():
    for X, Y in tqdm(test_loader, desc="Testing", leave=False):
        X, Y = X.to(device), Y.to(device)
        logits = model(X)
        loss = criterion(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))
        n_tokens = Y.numel()
        test_total_loss += loss.item() * n_tokens
        test_total_tokens += n_tokens

test_loss = test_total_loss / test_total_tokens
test_ppl = math.exp(test_loss)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test PPL: {test_ppl:.2f}")

Testing:   0%|          | 0/123 [00:00<?, ?it/s]

Test Loss: 5.0027
Test PPL: 148.82


Lưu kết quả cuối cùng

In [ ]:
summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "experiment": EXPERIMENT_NAME,
    "parameters": total_params,
    "context_length": CONTEXT_LENGTH,
    "batch_size": BATCH_SIZE,
    "best_valid_loss": result["best_valid_loss"],
    "best_valid_ppl": result["best_valid_ppl"],
    "test_loss": test_loss,
    "test_ppl": test_ppl,
    "epochs_trained": result["epochs_trained"],
    "training_time_seconds": result["training_time_seconds"]
}])

summary.to_csv(
    Path(result["results_dir"]) / "summary.csv",
    index=False
)

display(summary)

In [19]:
import json

from src.training.inference import (
    predict_next_token,
    generate_text
)

with open(
    root / "data" / "processed" / "word2idx.json",
    encoding="utf-8"
) as f:
    word2idx = json.load(f)

idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

model.load_state_dict(
    torch.load(
        r"C:\nam4\Nam4\deeplearning\baitapnhom\checkpoints\cnn\word_c64\best_model.pt",
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("Đã nạp best CNN model!")

Đã nạp best CNN model!


In [20]:
prompt = "The president of the"

predictions = predict_next_token(
    model=model,
    text=prompt,
    word2idx=word2idx,
    idx2word=idx2word,
    device=device,
    context_length=CONTEXT_LENGTH,
    top_k=5
)

print("Input:", prompt)
print("\nTop 5 predictions:")

for item in predictions:
    print(
        f"{item['token']:15s} "
        f"{item['probability'] * 100:.2f}%"
    )

Input: The president of the

Top 5 predictions:
first           1.40%
United          0.69%
song            0.65%
game            0.61%
same            0.61%


In [21]:
prompt = "The history of the"

generated = generate_text(
    model=model,
    prompt=prompt,
    word2idx=word2idx,
    idx2word=idx2word,
    device=device,
    context_length=CONTEXT_LENGTH,
    max_new_tokens=100,
    temperature=0.8,
    top_k=20
)

print("Prompt:", prompt)
print("\nGenerated text:")
print(generated)

Prompt: The history of the

Generated text:
The history of the end of a result of the city from the first time . She was the main first of the " A , with it is a small single @ - @ level . He was known to a few @ - @ inch of the song ' s " " , and " " the " " " " " of " " , " " " " " " " " " ' " , " . " " " " , " " " the . "
